# Language: Text Analytics in Action

In this notebook you will learn how to use **Azure AI Language** to:

1. **Detect the language** of a piece of text
2. **Extract key phrases** from a document
3. **Analyze sentiment** (positive / negative / neutral)
4. **Recognize named entities** (people, places, organizations, etc.)

---

### Prerequisites: Deploy the Azure Resource

Before running this notebook, you need an **Azure AI Language** resource. Instead of configuring it manually in the Azure Portal, deploy the included **ARM template** which creates the resource in the correct configuration automatically.

> **Tip:** Open `deploy-language.parameters.json` to customize the resource name, region, or SKU before deploying. The default SKU is **S** (paid). Change it to **F0** for the free tier.

**1. Deploy the resource** — run the cell below (replace `<your-resource-group>`):

In [ ]:
import subprocess, shutil

# ⚠️ Replace <your-resource-group> with your actual resource group name
RESOURCE_GROUP = "Foundry"

# Resolve the full path to the Azure CLI (needed for Windows where az is a .cmd file)
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "create",
     "--resource-group", RESOURCE_GROUP,
     "--template-file", "deploy-language.json",
     "--parameters", "deploy-language.parameters.json"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

**2. Assign the required RBAC role** so your identity can use the resource (replace `<your-resource-group>`):

> **Note:** RBAC role assignments can take **1–2 minutes** to propagate. If you get a 401 error on your first run, wait a moment and try again.

In [ ]:
import subprocess, shutil

RESOURCE_GROUP = "Foundry"
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

# Step A: Get the signed-in user's Object ID
user_result = subprocess.run(
    [AZ, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
user_id = user_result.stdout.strip()
print(f"Your User ID: {user_id}")

# Step B: Get the resource ID from the deployment output
scope_result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-language",
     "--query", "properties.outputs.resourceId.value", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
resource_id = scope_result.stdout.strip()
print(f"Resource ID:  {resource_id}")

# Step C: Assign the RBAC role
role_result = subprocess.run(
    [AZ, "role", "assignment", "create",
     "--assignee", user_id,
     "--role", "Cognitive Services User",
     "--scope", resource_id],
    capture_output=True, text=True, encoding="utf-8"
)
print(role_result.stdout)
if role_result.returncode != 0:
    print("ERROR:", role_result.stderr)

**3. Update the `.env` file** in this folder with the endpoint from the deployment output:

```
LANGUAGE_ENDPOINT="https://<your-account-name>.cognitiveservices.azure.com/"
```

You can retrieve this value by running the cell below (replace `<your-resource-group>`):

In [ ]:
import subprocess, shutil

RESOURCE_GROUP = "Foundry"
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-language",
     "--query", "properties.outputs"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

### Authentication

This notebook uses **Microsoft Entra ID** (`DefaultAzureCredential`) instead of API keys.
Make sure you are logged in with `az login`.

## Step 1: Install the Text Analytics SDK

In [ ]:
%pip install azure-ai-textanalytics azure-identity python-dotenv

## Step 2: Load Environment Variables and Create the Client

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.textanalytics import TextAnalyticsClient
from azure.identity import DefaultAzureCredential

load_dotenv(override=True)

language_endpoint = os.getenv("LANGUAGE_ENDPOINT")

# Authenticate via Entra ID (uses az login credentials)
credential = DefaultAzureCredential()

ta_client = TextAnalyticsClient(
    endpoint=language_endpoint,
    credential=credential,
)

print("Endpoint:", language_endpoint)
print("Client ready:", ta_client is not None)

## Step 3: Detect Language

Pass sentences in different languages and the service will tell you
which language each one is written in.

In [ ]:
multilingual_samples = [
    "Bonjour, comment allez-vous aujourd'hui ?",
    "Ich lerne gerne neue Programmiersprachen.",
    "La inteligencia artificial está cambiando el mundo.",
]

detection_results = ta_client.detect_language(documents=multilingual_samples)

for doc, result in zip(multilingual_samples, detection_results):
    print(f"Text: {doc}")
    print(f"  → Language: {result.primary_language.name} "
          f"(confidence: {result.primary_language.confidence_score:.0%})")
    print()

## Step 4: Extract Key Phrases

Key phrase extraction pulls out the most important terms from your text,
which is useful for indexing, tagging, or summarization.

In [ ]:
tech_paragraph = [
    "Microsoft Azure provides a wide range of cloud services including "
    "virtual machines, Kubernetes clusters, serverless functions, and "
    "managed databases that help organizations scale their infrastructure."
]

kp_results = ta_client.extract_key_phrases(documents=tech_paragraph)

print("Key phrases found:")
for phrase in kp_results[0].key_phrases:
    print(f"  • {phrase}")

## Step 5: Analyze Sentiment

Sentiment analysis classifies each sentence as **positive**, **negative**,
or **neutral**. This is invaluable for processing customer feedback,
reviews, or support tickets.

In [ ]:
review_samples = [
    "The new update is fantastic -- performance improved dramatically and the UI feels much smoother.",
    "I've been waiting three weeks for a response from support and still haven't heard back.",
    "The package arrived on Tuesday as scheduled.",
]

sentiment_results = ta_client.analyze_sentiment(documents=review_samples)

for result in sentiment_results:
    sentence = result.sentences[0]
    print(f"Sentiment: {sentence.sentiment.upper():>8}  │  {sentence.text}")

## Step 6: Recognize Named Entities

Named Entity Recognition (NER) identifies real-world objects such as
people, places, companies, dates, and monetary values.

In [ ]:
news_snippets = [
    "In January 2025, OpenAI announced a partnership with Microsoft "
    "to invest $10 billion in artificial intelligence research at "
    "their campus in San Francisco."
]

ner_results = ta_client.recognize_entities(documents=news_snippets)

print(f"{'Entity':<25} {'Category':<15} {'Confidence'}")
print("-" * 55)
for entity in ner_results[0].entities:
    print(f"{entity.text:<25} {entity.category:<15} {entity.confidence_score:.0%}")